In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "boto3", "pandas", "numpy", "scikit-learn", "folium"])

## Dependencies

# Notebook 1 — Stop Detection

**Technique:** State-machine speed thresholding + DBSCAN spatial clustering

**Goal:** Recover bus stop locations from raw GPS data without a reference schedule.

**Algorithm:**
1. Per vehicle: run a state machine — transition to STOPPED when `speed < 2 km/h` for ≥60s, back to MOVING when `speed > 5 km/h` (hysteresis avoids toggling)
2. Collect all stop (lat, lon) points
3. DBSCAN cluster recurring stops (eps=0.0005° ≈ 55m, min_samples=10)
4. Visualise on Folium map

Hysteresis prevents false toggles at slow-moving bus stops (1-2 km/h creep).

In [ ]:
import boto3, json, io
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
import folium
from botocore.client import Config

s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)
BUCKET = 'bus-history'

In [ ]:
def load_trip_segments(max_objects=500):
    records = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix='trips/'):
        for obj in page.get('Contents', [])[:max_objects]:
            body = s3.get_object(Bucket=BUCKET, Key=obj['Key'])['Body'].read()
            records.append(json.loads(body))
    return pd.DataFrame(records)

def load_raw_events_sample(max_objects=50):
    records = []
    paginator = s3.get_paginator('list_objects_v2')
    for page in paginator.paginate(Bucket=BUCKET, Prefix='2025-'):
        for obj in page.get('Contents', [])[:max_objects]:
            body = s3.get_object(Bucket=BUCKET, Key=obj['Key'])['Body'].read()
            records.extend(json.loads(body))
    return pd.DataFrame(records)

In [ ]:
def detect_stops(events_df, speed_thresh=2.0, restart_speed=5.0, min_duration_s=60):
    stops = []
    for vehicle, grp in events_df.groupby('vehicle'):
        grp = grp.sort_values('datetime').reset_index(drop=True)
        state = 'MOVING'
        stop_start = None
        stop_lat = stop_lon = None

        for _, row in grp.iterrows():
            spd = row.get('speed') if pd.notna(row.get('speed')) else 0

            if state == 'MOVING' and spd < speed_thresh:
                state = 'STOPPED'
                stop_start = row['datetime']
                stop_lat = row.get('y')
                stop_lon = row.get('x')

            elif state == 'STOPPED' and spd >= restart_speed:
                duration = row['datetime'] - stop_start
                if duration >= min_duration_s and stop_lat is not None:
                    stops.append({
                        'vehicle': vehicle,
                        'lat': stop_lat, 'lon': stop_lon,
                        'duration_s': duration,
                        'start_ts': stop_start, 'end_ts': row['datetime']
                    })
                state = 'MOVING'
    return pd.DataFrame(stops)

print('Functions defined.')

In [ ]:
print('Loading raw GPS events from MinIO...')
raw_df = load_raw_events_sample(max_objects=100)
print(f'Loaded {len(raw_df):,} events across {raw_df["vehicle"].nunique() if "vehicle" in raw_df.columns else 0} vehicles')
print(raw_df.head(3))

In [ ]:
stops_df = detect_stops(raw_df)
print(f'Detected {len(stops_df):,} stop events')
print(stops_df.describe())

In [ ]:
if len(stops_df) >= 10:
    coords = stops_df[['lat','lon']].dropna().values
    # eps 0.0005° ≈ 55m tại TPHCM
    db = DBSCAN(eps=0.0005, min_samples=10, algorithm='ball_tree', metric='haversine')
    stops_df = stops_df.dropna(subset=['lat','lon']).copy()
    stops_df['cluster'] = db.fit_predict(np.radians(stops_df[['lat','lon']].values))
    
    cluster_centers = stops_df[stops_df['cluster'] >= 0].groupby('cluster')[['lat','lon']].mean()
    print(f'DBSCAN found {len(cluster_centers)} recurring stop locations (noise: {(stops_df["cluster"]==-1).sum()})')
    print(cluster_centers.head(10))
else:
    print('Not enough stop events for DBSCAN — run with more data files loaded.')

In [ ]:
m = folium.Map(location=[10.7769, 106.7009], zoom_start=12, tiles='CartoDB dark_matter')

# tất cả điểm dừng
for _, row in stops_df.iterrows():
    if pd.notna(row.get('lat')) and pd.notna(row.get('lon')):
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=3, color='#4a90d9', fill=True, fill_opacity=0.4,
            popup=f"Vehicle: {row['vehicle'][:8]} | Duration: {row['duration_s']}s"
        ).add_to(m)

# tâm cụm nổi bật hơn
if 'cluster' in stops_df.columns:
    for _, centre in cluster_centers.iterrows():
        folium.CircleMarker(
            location=[centre['lat'], centre['lon']],
            radius=8, color='#e94560', fill=True, fill_opacity=0.9,
            popup='Recurring bus stop'
        ).add_to(m)

m.save('/tmp/stop_detection_map.html')
print('Map saved to /tmp/stop_detection_map.html')
m